In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split
from pathlib import Path

In [2]:
import dagshub
dagshub.init(repo_owner='AMR-ITH', repo_name='RealEstateInsights', mlflow=True)
import mlflow

Accessing as AMR-ITH

Initialized MLflow to track repo "AMR-ITH/RealEstateInsights"

Repository AMR-ITH/RealEstateInsights initialized!

In [3]:
# set the tracking server

mlflow.set_tracking_uri("https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow")

In [4]:
# mlflow experiment

mlflow.set_experiment("Exp 1 - simple model with and without target transformation")

<Experiment: artifact_location='mlflow-artifacts:/3228cb460f7c40c4996e00010eed0095', creation_time=1746439877628, experiment_id='1', last_update_time=1746439877628, lifecycle_stage='active', name='Exp 1 - simple model with and without target transformation', tags={}>

# Load the data

In [5]:
# pathlib is a module in Python that provides an object-oriented interface 
# for working with file system paths.
current = Path.cwd()
parent = current.parent

# load the dat 
df = pd.read_csv(parent /'data/interim_data.csv')
df.head()

df.drop(columns=['carpet_area','super_bulit_area','nearbylocation','facility','apartment_name','appartment_loc'], inplace=True)

In [6]:
# check for missing values

df.isnull().sum()

zone                      0
bhk_type                  0
construction_status       0
bulit_area                0
price_value               0
luxury_facility_scores    0
dtype: int64

# with out transforming the traget variable : price_value 

In [7]:
temp_df = df.copy()

X = temp_df.drop(columns=['price_value'])
y = temp_df['price_value']

In [8]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [9]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (4881, 5)
The shape of test data is (1221, 5)


In [10]:
# do the basic processing input data

num_cols = ['bulit_area','luxury_facility_scores']
nomial_cols = ['zone']
ordinal_cols = ['construction_status','bhk_type']


In [11]:
for col in ordinal_cols:
    print(col,":",X_train[col].unique())

construction_status : ['Moderatly Old' 'New Property' 'undefined' 'Relatively New'
 'Under Construction' 'Old']
bhk_type : [2 3 4 1]


In [12]:
construction_status_order = ['New Property','Under Construction', 'Relatively New', 'Moderatly Old', 'Old','undefined']
bhk_type_order = ['1','2','3','4','5','6','7','8','9','10']

In [13]:
# build a preprocessor

prepocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
        ("nominal_encode", OneHotEncoder(handle_unknown="ignore",sparse_output=False), nomial_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[construction_status_order,bhk_type_order]), ordinal_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

prepocessor.set_output(transform="pandas")

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(),
                                 ['bulit_area', 'luxury_facility_scores']),
                                ('nominal_encode',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['zone']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['New Property',
                                                             'Under '
                                                             'Construction',
                                                             'Relatively New',
                                                             'Moderatly Old',
                                                             'Old',
                                                             'undefined'],
                                                            ['1', '2', '3', '4',
                                                             '5', '6', '7', '8',
                                                             '9', '10']]),
                                 ['construction_status', 'bhk_type'])],
                  verbose_feature_names_out=False)

In [14]:
# transform the data

X_train_trans = prepocessor.fit_transform(X_train)
X_test_trans = prepocessor.transform(X_test)

X_train_trans

,bulit_area,luxury_facility_scores,zone_east,zone_north,zone_south,zone_west,construction_status,bhk_type
4472,0.115549,0.336323,0.0,0.0,1.0,0.0,3.0,1.0
4842,0.192971,0.183857,0.0,0.0,1.0,0.0,0.0,2.0
6026,0.227986,0.000000,0.0,0.0,0.0,1.0,5.0,2.0
485,0.144210,0.322870,1.0,0.0,0.0,0.0,0.0,1.0
3717,0.337440,0.089686,0.0,1.0,0.0,0.0,3.0,3.0
...,...,...,...,...,...,...,...,...
3772,0.271041,0.838565,0.0,1.0,0.0,0.0,1.0,3.0
5191,0.101673,0.502242,0.0,0.0,1.0,0.0,4.0,2.0
5226,0.169887,0.219731,0.0,0.0,1.0,0.0,0.0,2.0
5390,0.052502,0.596413,0.0,0.0,0.0,1.0,2.0,0.0


## Linear Regression model

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# linear regression model
lin_reg = LinearRegression()

lin_reg.fit(X_train_trans, y_train)

# get the prediction
y_pred_train = lin_reg.predict(X_train_trans)
y_pred_test = lin_reg.predict(X_test_trans)

# get the metrics r2,mse,mae
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train= mean_squared_error(y_train, y_pred_train)
r2_score_train= r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_score_test = r2_score(y_test, y_pred_test)


# metrics for linear regression model
print("Linear Regression Model")
print("R2 Train:",r2_score_train)
print("R2 Test:",r2_score_test)
print("MAE Train:",mae_train)
print("MAE Test:",mae_test)

Linear Regression Model
R2 Train: 0.6977094695969779
R2 Test: 0.6534825232210195
MAE Train: 0.432078988565175
MAE Test: 0.46807920331567504


In [16]:
# calculate the cross val score

from sklearn.model_selection import cross_val_score

scores_cv_linear_model = cross_val_score(lin_reg,X_train_trans,y_train,cv=5,scoring="r2",n_jobs=-1)

scores_cv_linear_model

array([0.71862626, 0.70507387, 0.67254651, 0.68176952, 0.69690913])

In [17]:
# log experiment
with mlflow.start_run(run_name="No Target tranformation-lr"):
    # mlflow log exp type
    mlflow.log_param("experiment_type", "no target transformation")
    # log model params
    mlflow.log_params(lin_reg.get_params())
    # log metrics
    mlflow.log_metric("train_r2", r2_score_train)
    mlflow.log_metric("test_r2", r2_score_test)
    mlflow.log_metric("train_mae", mae_train)   
    mlflow.log_metric("test_mae", mae_test)
    # log cross val score
    mlflow.log_metric("cross_val_score", scores_cv_linear_model.mean())

🏃 View run No Target tranformation-lr at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/1/runs/96fd19fdb0cc4b90bd670463499b1e11
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/1


## Random Forest Regressor Model

In [18]:
from sklearn.ensemble import RandomForestRegressor

# random forest model
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)

rf_reg.fit(X_train_trans, y_train)

# get the prediction
y_pred_train = rf_reg.predict(X_train_trans)
y_pred_test = rf_reg.predict(X_test_trans)

# get the metrics r2,mse,mae
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train= mean_squared_error(y_train, y_pred_train)
r2_score_train= r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_score_test = r2_score(y_test, y_pred_test)


# metrics for linear regression model
print("Random Forest Model")
print("R2 Train:",r2_score_train)
print("R2 Test:",r2_score_test)
print("MAE Train:",mae_train)
print("MAE Test:",mae_test)



Random Forest Model
R2 Train: 0.9683328475213231
R2 Test: 0.7263513894869028
MAE Train: 0.1302761881918273
MAE Test: 0.3722796885991334


In [19]:
scores_cv_rf_model = cross_val_score(rf_reg,X_train_trans,y_train,cv=5,scoring="r2",n_jobs=-1)

scores_cv_rf_model

array([0.79507501, 0.77405177, 0.76629615, 0.76667022, 0.7672088 ])

In [20]:
# feature importance plot

(
    pd.DataFrame(
        rf_reg.feature_importances_,
        index=X_train_trans.columns,
        columns=["Feature Importance"]
    )
    .sort_values(by="Feature Importance",ascending=False)
)

,Feature Importance
bulit_area,0.785233
luxury_facility_scores,0.121780
construction_status,0.038107
bhk_type,0.020567
zone_south,0.011557
zone_north,0.008468
zone_east,0.007474
zone_west,0.006813


In [21]:
# log experiment
with mlflow.start_run(run_name="No Target tranformation-rf"):
    # mlflow log exp type
    mlflow.log_param("experiment_type", "no target transformation")
    # log model params
    mlflow.log_params(rf_reg.get_params())
    # log metrics
    mlflow.log_metric("train_r2", r2_score_train)
    mlflow.log_metric("test_r2", r2_score_test)
    mlflow.log_metric("train_mae", mae_train)   
    mlflow.log_metric("test_mae", mae_test)
    # log cross val score
    mlflow.log_metric("cross_val_score", scores_cv_rf_model.mean())

🏃 View run No Target tranformation-rf at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/1/runs/00e1737580214c098b8d2e1691ad5acd
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/1


# with  transforming the traget variable : price_value 

In [22]:

from sklearn.preprocessing import PowerTransformer

# Setup power transformer
pt = PowerTransformer(method="yeo-johnson")
df["price_value_pt"] = pt.fit_transform(df[["price_value"]])



In [23]:
temp_df = df.copy()

X = temp_df.drop(columns=['price_value','price_value_pt'])
y = temp_df['price_value']  # Original target
y_pt = temp_df['price_value_pt']  # Transformed target

In [24]:
X_train, X_test, y_train, y_test, y_train_pt, y_test_pt = train_test_split(
    X, y, y_pt, test_size=0.2, random_state=42
)

In [25]:
# Check for NaN values in transformed features
print("NaN values in X_train_trans:", np.isnan(X_train_trans).sum())
print("NaN values in X_test_trans:", np.isnan(X_test_trans).sum())

NaN values in X_train_trans: bulit_area                0
luxury_facility_scores    0
zone_east                 0
zone_north                0
zone_south                0
zone_west                 0
construction_status       0
bhk_type                  0
dtype: int64
NaN values in X_test_trans: bulit_area                0
luxury_facility_scores    0
zone_east                 0
zone_north                0
zone_south                0
zone_west                 0
construction_status       0
bhk_type                  0
dtype: int64


In [26]:
# transform the data

X_train_trans = prepocessor.fit_transform(X_train)
X_test_trans = prepocessor.transform(X_test)

X_train_trans

,bulit_area,luxury_facility_scores,zone_east,zone_north,zone_south,zone_west,construction_status,bhk_type
4472,0.115549,0.336323,0.0,0.0,1.0,0.0,3.0,1.0
4842,0.192971,0.183857,0.0,0.0,1.0,0.0,0.0,2.0
6026,0.227986,0.000000,0.0,0.0,0.0,1.0,5.0,2.0
485,0.144210,0.322870,1.0,0.0,0.0,0.0,0.0,1.0
3717,0.337440,0.089686,0.0,1.0,0.0,0.0,3.0,3.0
...,...,...,...,...,...,...,...,...
3772,0.271041,0.838565,0.0,1.0,0.0,0.0,1.0,3.0
5191,0.101673,0.502242,0.0,0.0,1.0,0.0,4.0,2.0
5226,0.169887,0.219731,0.0,0.0,1.0,0.0,0.0,2.0
5390,0.052502,0.596413,0.0,0.0,0.0,1.0,2.0,0.0


In [27]:
pt.lambdas_

array([-0.90168691])

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Linear regression model
lin_reg = LinearRegression()

# Fit on transformed features and transformed target
lin_reg.fit(X_train_trans, y_train_pt)

# Get the predictions (in transformed scale)
y_pred_train_pt = lin_reg.predict(X_train_trans)
y_pred_test_pt = lin_reg.predict(X_test_trans)

# Convert predictions back to original scale
# Need to reshape to 2D array for inverse_transform
y_pred_train = pt.inverse_transform(y_pred_train_pt.reshape(-1, 1)).flatten()
y_pred_test = pt.inverse_transform(y_pred_test_pt.reshape(-1, 1)).flatten()

# Calculate metrics using original scale values
# Compare predictions with actual values in the same scale
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)
r2_score_train = r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_score_test = r2_score(y_test, y_pred_test)

# Print metrics for linear regression model
print("Linear Regression Model")
print("R2 Train:", r2_score_train)
print("R2 Test:", r2_score_test)
print("MAE Train:", mae_train)
print("MAE Test:", mae_test)
print("MSE Train:", mse_train)
print("MSE Test:", mse_test)
print("RMSE Train:", np.sqrt(mse_train))
print("RMSE Test:", np.sqrt(mse_test))

c:\Python\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
c:\Python\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(


ValueError: Input contains NaN.

yeo-johnson method data is highly skewed with extreme outliers.
inverse transform and leads to NaN, inf, or totally unrealistic values
doing other quantile transform methods 

## quantile method

In [29]:
from sklearn.preprocessing import QuantileTransformer

# Setup QuantileTransformer (output distribution set to 'normal')
qt = QuantileTransformer(output_distribution='normal', random_state=42)
df["price_value_pt"] = qt.fit_transform(df[["price_value"]])

temp_df = df.copy()

X = temp_df.drop(columns=['price_value','price_value_pt'])
y = temp_df['price_value']  # Original target
y_pt = temp_df['price_value_pt']  # Transformed target

X_train, X_test, y_train, y_test, y_train_pt, y_test_pt = train_test_split(
    X, y, y_pt, test_size=0.2, random_state=42
)

X_train_trans = prepocessor.fit_transform(X_train)
X_test_trans = prepocessor.transform(X_test)




In [30]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Linear regression model
lin_reg = LinearRegression()

# Fit on transformed features and transformed target
lin_reg.fit(X_train_trans, y_train_pt)



LinearRegression()

In [31]:
# Get the predictions (in transformed scale)
y_pred_train_pt = lin_reg.predict(X_train_trans)
y_pred_test_pt = lin_reg.predict(X_test_trans)

# Convert predictions back to original scale
y_pred_train = qt.inverse_transform(pd.DataFrame(y_pred_train_pt, columns=["price_value"])).flatten()
y_pred_test = qt.inverse_transform(pd.DataFrame(y_pred_test_pt, columns=["price_value"])).flatten()

# Calculate metrics using original scale values
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)
r2_score_train = r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_score_test = r2_score(y_test, y_pred_test)

# Print metrics
print("Linear Regression Model")
print("R2 Train:", r2_score_train)
print("R2 Test:", r2_score_test)
print("MAE Train:", mae_train)
print("MAE Test:", mae_test)
print("MSE Train:", mse_train)
print("MSE Test:", mse_test)
print("RMSE Train:", np.sqrt(mse_train))
print("RMSE Test:", np.sqrt(mse_test))

Linear Regression Model
R2 Train: 0.6817514412704313
R2 Test: 0.6282999144577047
MAE Train: 0.41429455871673065
MAE Test: 0.44034134271457337
MSE Train: 0.3625054918349665
MSE Test: 0.42588344822667434
RMSE Train: 0.6020842896430421
RMSE Test: 0.6525974626265982


In [32]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Linear regression model
rf_reg = RandomForestRegressor()

# Fit on transformed features and transformed target
rf_reg.fit(X_train_trans, y_train_pt)

RandomForestRegressor()

In [33]:
# Get the predictions (in transformed scale)
y_pred_train_pt = rf_reg.predict(X_train_trans)
y_pred_test_pt = rf_reg.predict(X_test_trans)

# Convert predictions back to original scale
y_pred_train = qt.inverse_transform(pd.DataFrame(y_pred_train_pt, columns=["price_value"])).flatten()
y_pred_test = qt.inverse_transform(pd.DataFrame(y_pred_test_pt, columns=["price_value"])).flatten()

# Calculate metrics using original scale values
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)
r2_score_train = r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_score_test = r2_score(y_test, y_pred_test)

# Print metrics
print("Linear Regression Model")
print("R2 Train:", r2_score_train)
print("R2 Test:", r2_score_test)
print("MAE Train:", mae_train)
print("MAE Test:", mae_test)
print("MSE Train:", mse_train)
print("MSE Test:", mse_test)
print("RMSE Train:", np.sqrt(mse_train))
print("RMSE Test:", np.sqrt(mse_test))

Linear Regression Model
R2 Train: 0.9612813099399042
R2 Test: 0.7004063452530593
MAE Train: 0.13391543899008698
MAE Test: 0.3783533781631702
MSE Train: 0.04410306786453502
MSE Test: 0.34326593862443516
RMSE Train: 0.2100073043123382
RMSE Test: 0.5858890156202241


**Conculsion** : Both the with target transformation or with out target transformation the model is giving similar results. So we can go with the model without target transformation.